# Poland Temperature - SARIMA Step-by-Step Analysis

**Goal**: Build and compare SARIMA models for Poland temperature forecasting

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
print("Libraries loaded!")

## 1. Load Data

In [ ]:
# Load Poland time series
df = pd.read_csv('/mnt/user-data/outputs/poland_timeseries.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')

print(f"Dataset: {len(df)} observations")
print(f"Period: {df['year'].min()} to {df['year'].max()}")
print(f"Mean temp: {df['AverageTemperatureCelsius'].mean():.2f}°C")
df.head()

## 2. Visualize Complete Series

In [ ]:
plt.figure(figsize=(15, 4))
plt.plot(df.index, df['AverageTemperatureCelsius'], linewidth=0.5)
plt.title('Poland Temperature 1743-2013', fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Temperature (°C)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Zoom: Last 5 Years (See Seasonality)

In [ ]:
last_5_years = df[df['year'] >= df['year'].max() - 4]

plt.figure(figsize=(15, 4))
plt.plot(last_5_years.index, last_5_years['AverageTemperatureCelsius'], 
         marker='o', markersize=4, linewidth=2)
plt.title('Last 5 Years - Clear Monthly Seasonality', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Observation: Strong 12-month seasonal pattern (peaks in summer, troughs in winter)")

## 4. Decomposition (Trend + Seasonal + Residual)

In [ ]:
# Use last 10 years for clearer visualization
recent = df[df['year'] >= df['year'].max() - 9]
decomp = seasonal_decompose(recent['AverageTemperatureCelsius'], 
                            model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(15, 8))
decomp.observed.plot(ax=axes[0], title='Original')
decomp.trend.plot(ax=axes[1], title='Trend')
decomp.seasonal.plot(ax=axes[2], title='Seasonal')
decomp.resid.plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.show()

print("Findings:")
print("✓ TREND: Slight upward (global warming)")
print("✓ SEASONAL: Regular 12-month cycle")
print("✓ => Need SARIMA to handle both!")

## 5. Stationarity Test (ADF)

In [ ]:
def test_stationarity(series, name):
    result = adfuller(series.dropna())
    print(f"{name}:")
    print(f"  ADF Statistic: {result[0]:.4f}")
    print(f"  p-value: {result[1]:.4f}")
    if result[1] < 0.05:
        print(f"  => STATIONARY (p<0.05)")
    else:
        print(f"  => NON-STATIONARY (need differencing)")
    print()

test_stationarity(df['AverageTemperatureCelsius'], "Original Series")
test_stationarity(df['AverageTemperatureCelsius'].diff(), "After 1st Difference")

## 6. ACF & PACF (Choose Parameters)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 6))

plot_acf(df['AverageTemperatureCelsius'].dropna(), lags=50, ax=axes[0,0])
axes[0,0].set_title('ACF - Original')

plot_pacf(df['AverageTemperatureCelsius'].dropna(), lags=50, ax=axes[0,1])
axes[0,1].set_title('PACF - Original')

plot_acf(df['AverageTemperatureCelsius'].diff().dropna(), lags=50, ax=axes[1,0])
axes[1,0].set_title('ACF - Differenced')

plot_pacf(df['AverageTemperatureCelsius'].diff().dropna(), lags=50, ax=axes[1,1])
axes[1,1].set_title('PACF - Differenced')

plt.tight_layout()
plt.show()

print("Observations:")
print("- Spikes at lags 12, 24, 36 => seasonal (s=12)")
print("- Suggests starting with SARIMA(1,1,1)(1,1,1,12)")

## 7. Train/Test Split

In [ ]:
# 80/20 split
split_idx = int(len(df) * 0.8)
train = df['AverageTemperatureCelsius'][:split_idx]
test = df['AverageTemperatureCelsius'][split_idx:]

print(f"Train: {len(train)} obs")
print(f"Test: {len(test)} obs")

plt.figure(figsize=(15, 4))
plt.plot(train.index, train, label='Train', alpha=0.7)
plt.plot(test.index, test, label='Test', alpha=0.7)
plt.axvline(train.index[-1], color='r', linestyle='--', label='Split')
plt.legend()
plt.title('Train/Test Split')
plt.tight_layout()
plt.show()

## 8. Manual SARIMA(1,1,1)(1,1,1,12)

In [ ]:
print("Fitting SARIMA(1,1,1)(1,1,1,12)...")

model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12),
                enforce_stationarity=False, enforce_invertibility=False)
fit = model.fit(disp=False)

print("Done!\n")
print(fit.summary())

In [ ]:
# Forecast and evaluate
forecast = fit.forecast(steps=len(test))

# Use .values to avoid pandas index issues
rmse = np.sqrt(np.mean((test.values - forecast.values)**2))
mae = np.mean(np.abs(test.values - forecast.values))

print(f"Performance:")
print(f"  RMSE: {rmse:.4f}°C")
print(f"  MAE: {mae:.4f}°C")
print(f"  AIC: {fit.aic:.2f}")

In [ ]:
# Plot forecast
plt.figure(figsize=(15, 5))
plt.plot(test.index, test.values, label='Actual', marker='o', markersize=3, linewidth=2)
plt.plot(test.index, forecast.values, label=f'Forecast (RMSE={rmse:.3f})', 
         linewidth=2, linestyle='--')
plt.title('SARIMA(1,1,1)(1,1,1,12) Forecast', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Try Other Configurations

In [ ]:
configs = [
    ((2, 1, 2), (1, 1, 1, 12)),
    ((0, 1, 1), (0, 1, 1, 12)),
    ((1, 1, 1), (2, 1, 1, 12)),
]

results = [{'config': '(1,1,1)(1,1,1,12)', 'rmse': rmse, 'aic': fit.aic}]

for order, seasonal in configs:
    try:
        print(f"Trying SARIMA{order}x{seasonal}...")
        m = SARIMAX(train, order=order, seasonal_order=seasonal,
                    enforce_stationarity=False, enforce_invertibility=False)
        f = m.fit(disp=False)
        fc = f.forecast(steps=len(test))
        r = np.sqrt(np.mean((test.values - fc.values)**2))
        results.append({'config': f'{order}x{seasonal}', 'rmse': r, 'aic': f.aic})
        print(f"  RMSE: {r:.4f}")
    except Exception as e:
        print(f"  Failed: {e}")

results_df = pd.DataFrame(results).sort_values('rmse')
print("\nComparison:")
print(results_df.to_string(index=False))
print(f"\n✓ Best: {results_df.iloc[0]['config']}")

## 10. Auto SARIMA

In [ ]:
try:
    from pmdarima import auto_arima
except:
    print("Installing pmdarima...")
    import subprocess
    subprocess.run(['pip', 'install', 'pmdarima', '--break-system-packages', '-q'])
    from pmdarima import auto_arima

print("Running auto_arima (takes 2-3 minutes)...\n")

auto = auto_arima(train, seasonal=True, m=12,
                  max_p=3, max_q=3, max_P=2, max_Q=2,
                  max_d=2, max_D=1,
                  trace=True, stepwise=True,
                  suppress_warnings=True)

print(f"\nBest: SARIMA{auto.order}x{auto.seasonal_order}")
print(f"AIC: {auto.aic():.2f}")

In [ ]:
auto_fc = auto.predict(n_periods=len(test))
auto_rmse = np.sqrt(np.mean((test.values - auto_fc)**2))

print(f"Auto SARIMA Performance:")
print(f"  RMSE: {auto_rmse:.4f}°C")

plt.figure(figsize=(15, 5))
plt.plot(test.index, test.values, label='Actual', marker='o', markersize=3)
plt.plot(test.index, auto_fc, label=f'Auto SARIMA (RMSE={auto_rmse:.3f})', 
         linewidth=2, linestyle='--')
plt.title(f'Auto SARIMA{auto.order}x{auto.seasonal_order}', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. SARIMAX (with Exogenous Variables)

In [ ]:
# Create exogenous variables
df_exog = df.copy()
df_exog['year_norm'] = (df_exog['year'] - df_exog['year'].min()) / 100  # normalize
df_exog['month_sin'] = np.sin(2 * np.pi * df_exog['month'] / 12)
df_exog['month_cos'] = np.cos(2 * np.pi * df_exog['month'] / 12)

exog_train = df_exog[['year_norm', 'month_sin', 'month_cos']][:split_idx]
exog_test = df_exog[['year_norm', 'month_sin', 'month_cos']][split_idx:]

print("Exogenous variables:")
print(exog_train.head())

In [ ]:
print("Fitting SARIMAX(1,1,1)(1,1,1,12) with exog...\n")

sarimax = SARIMAX(train, exog=exog_train,
                  order=(1,1,1), seasonal_order=(1,1,1,12),
                  enforce_stationarity=False, enforce_invertibility=False)
sarimax_fit = sarimax.fit(disp=False)

print("Done!")
print(sarimax_fit.summary())

In [ ]:
sarimax_fc = sarimax_fit.forecast(steps=len(test), exog=exog_test)
sarimax_rmse = np.sqrt(np.mean((test.values - sarimax_fc.values)**2))

print(f"SARIMAX Performance:")
print(f"  RMSE: {sarimax_rmse:.4f}°C")
print(f"  AIC: {sarimax_fit.aic:.2f}")

## 12. Final Comparison

In [ ]:
final = pd.DataFrame({
    'Model': [
        'SARIMA(1,1,1)(1,1,1,12)',
        f'Auto {auto.order}x{auto.seasonal_order}',
        'SARIMAX with exog'
    ],
    'RMSE': [rmse, auto_rmse, sarimax_rmse]
}).sort_values('RMSE')

print("\nFINAL RESULTS:")
print("="*50)
print(final.to_string(index=False))
print("\n🏆 Winner:", final.iloc[0]['Model'])

In [ ]:
# Save best model plot
plt.figure(figsize=(15, 5))
plt.plot(test.index, test.values, label='Actual', linewidth=2.5, color='black')
plt.plot(test.index, forecast.values, label=f'SARIMA RMSE={rmse:.3f}', 
         linewidth=2, linestyle='--', alpha=0.7)
plt.plot(test.index, auto_fc, label=f'Auto RMSE={auto_rmse:.3f}',
         linewidth=2, linestyle='--', alpha=0.7)
plt.plot(test.index, sarimax_fc.values, label=f'SARIMAX RMSE={sarimax_rmse:.3f}',
         linewidth=2, linestyle='--', alpha=0.7)
plt.title('Poland - All SARIMA Models', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/poland_sarima_comparison.png', dpi=150)
plt.show()

print("\n✓ Saved to poland_sarima_comparison.png")